# JHDN7CF1C03X5

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without resetting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged
%store -r restaurants_by_4m_coverage
%store -r time_differences
%store -r time_differences_details
%store -r before_after_details_true

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

Time Differences

In [ ]:
%store -r restaurant_data_unprocessed
timezones_acronyms = {}
for loc_id, df in restaurant_data_unprocessed.items():
    time = df['created_at'].iloc[0]
    timezone = time.strip('0123456789-+: ')
    timezones_acronyms[loc_id] = timezone
timezones = {
    '0RJH3FFPYBPEY': 'America/New_York',
    '1SQPTEGYPH0GA': 'America/Denver',
    '3AXDVZJYN9DRS': 'Europe/London',
    '75WYSXR9QBK5M': 'Pacific/Honolulu',
    '78AY09MVJVTYE': 'America/New_York',
    '9XKJD8DQTH559': 'America/New_York',
    'AQD04SM0J92WA': 'America/Los_Angeles',
    'CB2KHY1C2G9PT': 'America/New_York',
    'EMBVNVD207CC6': 'America/New_York',
    'JHDN7CF1C03X5': 'America/Chicago',
    'L3XS7WSJ4AJA3': 'Europe/London',
    'L69HYJ4Y3TR91': 'America/New_York',
    'LBMCPAYT7W36V': 'America/New_York',
    'LBZEEFSBJNB3Z': 'America/Los_Angeles',
    'LFZFT3VASXPED': 'Australia/Sydney',
    'LQ5EH4BKGV61T': 'America/New_York',
    'LZ5MR1TS37E7W': 'America/Los_Angeles',
    'MS8R16DY0JQAM': 'America/Los_Angeles',
    'N0PC58FB2XAZ3': 'America/Chicago',
    'S8MT0YGD2KTN9': 'America/New_York',
    'SAFK7ND1HR6XS': 'America/Los_Angeles',
    'SRQS8F7JWA9MZ': 'America/New_York',
    'V3Q26BHF3SE2H': 'America/New_York',
    'W8T41JZK0ZMEP': 'America/New_York',
    'WJA3YCD4QBWRX': 'America/New_York',
    '1G5AJ17XCH2A8': 'America/Chicago',
    'ADPFRN3QZRCXK': 'America/Los_Angeles',
    'ED5J990H5VAZT': 'America/Los_Angeles',
    '2HRX9P6HKXA8V': 'America/Los_Angeles',
    'C0BE4NDSW26QN': 'America/New_York'
}
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])

before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'] = before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'].str.title()

In [ ]:
loc_id = 'JHDN7CF1C03X5'
df = sales_and_menu_data[loc_id]

In [ ]:
print(df['item_name'].value_counts().to_string())

In [ ]:
print(df.query('item_name.str.contains("Beyond Burger")')['item_name'].value_counts().to_string())

In [ ]:
df = df.assign(item_modifications = lambda df: df['item_modifications'].str.replace("Un'Chicken|Un'chicken", 'Unchicken', regex=True))
df.query('item_modifications.str.contains("Unchicken")')

In [ ]:
['Avocado Dream',
'Veggie Portobello',
'Kale Vegetarian Wrap',
'Super Veg',
'Mediterranean Spinach Salad',
'Asian Sesame Salad',
'Quesadilla Chicken Fiesta',
'Healthy Start Wrap',
'Kale Caesar Salad',
'Santa Fe Quesadilla',
'Portobello Balsamic Toast']

In [ ]:
print(df.query('~dish_category.isin(["Merch","Drink","Alcohol"]) and is_plant_based == "Yes"')['item_name'].value_counts().to_string())

In [ ]:
df.query('~dish_category.isin(["Merch","Drink","Alcohol"]) and is_plant_based == "No"')['item_name'].value_counts()


In [ ]:
name_changes = {

}

# Swap the keys and values
name_changes_dict = {variant: canonical for canonical, variants in name_changes.items() for variant in variants}

# Item names to swap based on modications
modification_name_changes = [('Fresh Beyond Burger', 'Bacon', 'Beyond Burger With Bacon') ,
                             ('Fresh Beyond Burger', 'Cheddar Cheese', 'Beyond Burger With Dairy'), # sauce????
                             ('Beyond Burger Combo', 'Bacon', 'Beyond Burger Combo With Bacon'),
                             ('Beyond Burger Combo', 'Cheddar Cheese', 'Beyond Burger Combo With Dairy'),]

# Turn into dataframe for viewing
modification_name_changes_df = pd.DataFrame(data = modification_name_changes, columns = ['name', 'modification', 'new_name'])

non_alcoholic_drinks = []

alcoholic_drinks = []

merch = []

rare =[]

unknown = []

vegetarian = ['Beyond Burger With Dairy', 'Beyond Burger Combo With Dairy']

vegan = ['Fresh Beyond Burger','Beyond Burger Combo']

# Swap the keys and values
replacement_dict = {variant: canonical for canonical, variants in name_changes.items() for variant in variants}

# Items to remove
items_to_remove = []

In [ ]:
df_cleaned = (df
              .assign(item_name=lambda df: df['item_name']
                      .str.strip('123456789./\\ ')  # Clean up item names
                      .replace(replacement_dict)    # Replace names based on dictionary
                      #.replace(items_to_remove, pd.NA)  # Replace non-dish items with NA
              )
              #.dropna(subset=['item_name'])
              .assign(item_name = lambda df: np.select(condlist = [df['item_name'].eq(name) &  
                                                                   df['item_modifications'].str.contains(modification) for name, modification, _ in modification_name_changes],
                                                       choicelist = modification_name_changes_df['new_name'].tolist(),
                                                       default = df['item_name']),
                      dish_category = lambda df: df['dish_category']
                              .mask(df['item_name'].isin(alcoholic_drinks), 'Alcohol')
                              .mask(df['item_name'].isin(merch), 'Merch')
                              .mask(df['item_name'].isin(non_alcoholic_drinks), 'Drink'),
                      vegetarian = lambda df: df['item_name'].isin(vegetarian + vegan + non_alcoholic_drinks + alcoholic_drinks),
                      vegan = lambda df: df['item_name'].isin(vegan + non_alcoholic_drinks + alcoholic_drinks)
                )
              #.drop('unique_id', axis=1)
             )

df = df_cleaned
food_df = df.query('~dish_category.isin(["Alcohol", "Drink", "Merch"])')

In [ ]:
# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert('UTC')

top_n = 30

unique_dishes = (food_df
                 ['item_name']
                 .value_counts()
                 .to_frame(name='c')
                 [:top_n]
                 .index[::-1]
                 )

legend_handles = []

for dish in unique_dishes:
    
    dish_df = food_df.query('item_name == @dish')
    
    vmin = dish_df['unit_price'].min()
    vmax = dish_df['unit_price'].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.ScalarMappable(norm=norm, cmap='magma')
    
    weekly_quantities = (dish_df
                         .resample('W')
                         .agg({'item_quantity': 'sum', 'unit_price': 'mean'})
                         .query('0 < item_quantity')
                         .assign(week = lambda df: df.index.tz_localize(None).to_period('W'))
                         .set_index('week')
                         )
    
    # For every active week
    for week, row  in weekly_quantities.iterrows():
        
        weekly_quantity = row['item_quantity']
        dot_size = weekly_quantity/10 + 3
        weekly_price = row['unit_price']
        color = cmap.to_rgba(weekly_price)

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors=color, lw=dot_size, label=loc_id)
        
    ax.text(x=food_df.index[-1] + pd.DateOffset(100), y=dish, s=f'${vmin/100:.2f}-${vmax/100:.2f}', verticalalignment='center', horizontalalignment='left', fontsize='x-small', color='gray')
    
    # Create a custom legend entry for this dish
    #color_patch_min = mpatches.Patch(color=cmap.to_rgba(vmin), label=f'{dish} Min: ${vmin/100:.2f}')
    #color_patch_max = mpatches.Patch(color=cmap.to_rgba(vmax), label=f'{dish} Max: ${vmax/100:.2f}')
    #legend_handles.extend([color_patch_min, color_patch_max])

# Place a red circle for the promotional item
ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title(f'Weekly Sales of Top {top_n} Dishes for {loc_id}')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')
#ax.legend(handles=legend_handles, title="Price Range per Dish", fontsize='small', loc='upper left', bbox_to_anchor=(1, 1))

# Figure
introduction_fig.tight_layout(rect=[0, 0, 0.85, 1])

plt.show()

In [ ]:
df.query('~vegetarian')['unit_price'].mean()


In [ ]:
df.query('is_plant_based == "Yes"')['item_name'].nunique() / df['item_name'].nunique()